In [ ]:
import os
import csv
import ast
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import colorcet as cc
import matplotlib.colors as mcolors
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from utils import savefig

plt.rcParams['font.size'] = 14

mpl.rcParams['svg.fonttype'] = 'none'


In [ ]:
def load_data(path, model_num):
    data_single_model = {}
    data_single_model["model_num"] = model_num

    classifier_data = pickle.load(open(path/"ridge_classifier_stat.pkl", "rb"))
    data_single_model["index_decoding_accuracy_encoding_phase"] = classifier_data["index_enc_acc"]
    data_single_model["item_decoding_accuracy_encoding_phase"] = classifier_data["item_enc_acc"]
    data_single_model["last_item_decoding_accuracy_encoding_phase"] = classifier_data["item_enc_acc_last"]
    data_single_model["index_decoding_accuracy_recall_phase"] = classifier_data["index_rec_acc"]
    data_single_model["item_decoding_accuracy_recall_phase"] = classifier_data["item_rec_acc"]
    data_single_model["last_item_decoding_accuracy_recall_phase"] = classifier_data["item_rec_acc_last"]
    data_single_model["index_decoding_accuracy"] = (data_single_model["index_decoding_accuracy_encoding_phase"] + data_single_model["index_decoding_accuracy_recall_phase"]) / 2
    data_single_model["item_decoding_accuracy"] = (data_single_model["item_decoding_accuracy_encoding_phase"] + data_single_model["item_decoding_accuracy_recall_phase"]) / 2
    data_single_model["last_item_decoding_accuracy"] = (data_single_model["last_item_decoding_accuracy_encoding_phase"] + data_single_model["last_item_decoding_accuracy_recall_phase"]) / 2

    explained_variance_data = np.load(path/"explained_variance.npy")
    data_single_model["explained_variance_encoding_index"] = explained_variance_data[0]
    data_single_model["explained_variance_recall_index"] = explained_variance_data[1]
    data_single_model["explained_variance_index"] = (data_single_model["explained_variance_encoding_index"] + data_single_model["explained_variance_recall_index"]) / 2
    data_single_model["explained_variance_encoding_identity"] = explained_variance_data[2]
    data_single_model["explained_variance_recall_identity"] = explained_variance_data[3]
    data_single_model["explained_variance_identity"] = (data_single_model["explained_variance_encoding_identity"] + data_single_model["explained_variance_recall_identity"]) / 2

    cross_decoding_data = np.load(path/"cross_acc.npy")
    data_single_model["cross_decoding_accuracy_index_rec_enc"] = cross_decoding_data[0]
    data_single_model["cross_decoding_accuracy_identity_rec_enc"] = cross_decoding_data[1]
    data_single_model["cross_decoding_accuracy_index_enc_rec"] = cross_decoding_data[2]
    data_single_model["cross_decoding_accuracy_identity_enc_rec"] = cross_decoding_data[3]
    data_single_model["cross_decoding_accuracy_index"] = (data_single_model["cross_decoding_accuracy_index_rec_enc"] + data_single_model["cross_decoding_accuracy_index_enc_rec"]) / 2
    data_single_model["cross_decoding_accuracy_identity"] = (data_single_model["cross_decoding_accuracy_identity_rec_enc"] + data_single_model["cross_decoding_accuracy_identity_enc_rec"]) / 2

    return data_single_model


In [ ]:
data_condfr = []

data_folder = Path("./experiments/CondFR/figures/free_recall/ValueMemoryGRU")
seq_len = 8

setups = ["setup_condfr_fixone_noise2"]

for setup in setups:
    for i in range(5):
        data_path = data_folder / setup / str(i)
        if os.path.exists(data_path/"cross_acc.npy"):
            data_single_model = load_data(data_path, i)
            if data_single_model:
                data_single_model["pretrained"] = False
                data_condfr.append(data_single_model)

df_condfr = pd.DataFrame(data_condfr)
print(len(df_condfr))

In [ ]:
explained_variance_index_mean = df_condfr["explained_variance_index"].mean()
explained_variance_index_std = df_condfr["explained_variance_index"].std()
print(explained_variance_index_mean, explained_variance_index_std)

explained_variance_identity_mean = df_condfr["explained_variance_identity"].mean()
explained_variance_identity_std = df_condfr["explained_variance_identity"].std()
print(explained_variance_identity_mean, explained_variance_identity_std)

explained_variance_index_data = df_condfr["explained_variance_index"].values
explained_variance_identity_data = df_condfr["explained_variance_identity"].values


# Create a bar plot for the explained variance data
labels = ['Index', 'Identity']
means = [explained_variance_index_mean, explained_variance_identity_mean]
stds = [explained_variance_index_std, explained_variance_identity_std]

x = range(len(labels))

fig = plt.figure(figsize=(2.5, 3.3), dpi=180)
ax = plt.gca()
plt.bar(x, means, 0.8, yerr=stds, align='center', color=["#A1C181", "#62B6CB"], capsize=5)
plt.scatter(np.zeros_like(explained_variance_index_data)+np.random.uniform(-0.2, 0.2, size=len(explained_variance_index_data)), explained_variance_index_data, color="grey", alpha=0.5, s=30)
plt.scatter(np.ones_like(explained_variance_identity_data)+np.random.uniform(-0.2, 0.2, size=len(explained_variance_identity_data)), explained_variance_identity_data, color="grey", alpha=0.5, s=30)
plt.ylabel('explained variance')
plt.xlabel('variables')
ax.set_xticks([0,1])
ax.set_xticklabels(labels, fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
savefig("./figures/fig6", "fig6E", format="svg", close=False)
plt.show()



In [ ]:
cross_decoding_accuracy_index_mean = df_condfr["cross_decoding_accuracy_index"].mean()
cross_decoding_accuracy_index_std = df_condfr["cross_decoding_accuracy_index"].std()
print(cross_decoding_accuracy_index_mean, cross_decoding_accuracy_index_std)

cross_decoding_accuracy_identity_mean = df_condfr["cross_decoding_accuracy_identity"].mean()
cross_decoding_accuracy_identity_std = df_condfr["cross_decoding_accuracy_identity"].std()
print(cross_decoding_accuracy_identity_mean, cross_decoding_accuracy_identity_std)


# Create a bar plot for the explained variance data
labels = ['Index', 'Identity']
means = [cross_decoding_accuracy_index_mean, cross_decoding_accuracy_identity_mean]
stds = [cross_decoding_accuracy_index_std, cross_decoding_accuracy_identity_std]

x = range(len(labels))

fig = plt.figure(figsize=(1.8, 3.3), dpi=180)
ax = plt.gca()
plt.bar(x, means, 0.8, yerr=stds, align='center', color=["#A1C181", "#62B6CB"], capsize=5)
plt.ylabel('cross-phase\ndecoding accuracy')
ax.set_xticks([0,1])
ax.set_xticklabels(labels, fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()



### key-value memory model

In [ ]:
data_folder = Path("/scratch/ml8736/memory-replay/Transform/figures/ValueMemoryGRU/setup_kv_sametransform")

keys_index_variance, keys_identity_variance, values_index_variance, values_identity_variance = [], [], [], []

for i in range(20):
    try:
        with open(data_folder / str(i) / "contiguity_effect.csv", "r") as f:
            reader = csv.reader(f)
            for row in reader:
                accuracy = float(row[0])
                break
    except:
        continue
    if accuracy < 0.6:
        continue
    try:
        data = np.load(data_folder / str(i) / "explained_variance.npy")
        if data[0] < 0.6:
            continue
        keys_index_variance.append(data[0])
        keys_identity_variance.append(data[1])
        values_index_variance.append(data[2])
        values_identity_variance.append(data[3])
    except:
        pass

keys_index_variance = np.array(keys_index_variance)
keys_identity_variance = np.array(keys_identity_variance)
values_index_variance = np.array(values_index_variance)
values_identity_variance = np.array(values_identity_variance)

print(keys_index_variance)
print(keys_identity_variance)
print(values_index_variance)
print(values_identity_variance)


In [ ]:
plt.figure(figsize=(4, 3.3), dpi=300)
bar_width = 0.35
index = np.arange(2)
plt.bar(index, [np.mean(keys_index_variance), np.mean(values_index_variance)], bar_width, label="index", color=["#9A8C98"],
    yerr=[np.std(keys_index_variance), np.std(values_index_variance)])
plt.bar(index + bar_width, [np.mean(keys_identity_variance), np.mean(values_identity_variance)], bar_width, label="identity", color=["#C9ADA7"],
    yerr=[np.std(keys_identity_variance), np.std(values_identity_variance)])
plt.scatter(np.ones_like(keys_index_variance)*0+np.random.uniform(-0.1, 0.1, size=len(keys_index_variance)), keys_index_variance, color="grey", alpha=0.5, s=20)
plt.scatter(np.ones_like(keys_identity_variance)*bar_width+np.random.uniform(-0.1, 0.1, size=len(keys_identity_variance   )), keys_identity_variance, color="grey", alpha=0.5, s=20)
plt.scatter(np.ones_like(values_index_variance)+np.random.uniform(-0.1, 0.1, size=len(values_index_variance)), values_index_variance, color="grey", alpha=0.5, s=20)
plt.scatter(np.ones_like(values_identity_variance)*(1+bar_width)+np.random.uniform(-0.1, 0.1, size=len(values_identity_variance)), values_identity_variance, color="grey", alpha=0.5, s=20)
plt.xlabel("variable")
plt.ylabel("explained variance")
plt.xticks(index + bar_width / 2, ["keys", "values"])
plt.legend(frameon=False)
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
savefig("./figures/exfig1", "exfig1C", format="svg", close=False)
plt.show()
